# Notebook 23 — Graph Embeddings of Residue-Lane Manifolds

**Repo:** `mod30-residue-lanes`  
**Layer:** `notebooks/`

Notebook 19 studied temporal spectral dynamics across rolling residue manifolds.

Notebook 23 treats the eight admissible residue lanes as graph nodes and rolling prime-lane correlations as weighted edges.

Constraint view:

> Residue lanes become a graph manifold when rolling prime-lane trajectories define weighted relationships between lanes.


## Goals

1. Detect repo root and create standard output directories.
2. Generate rolling prime-filtered lane vectors.
3. Construct lane similarity, covariance, and adjacency matrices.
4. Build weighted graph nodes and edges.
5. Compute graph Laplacian and Laplacian embedding.
6. Measure weighted degree, centrality, and strongest-neighbor structure.
7. Project rolling lane signals onto graph eigenmodes.
8. Export CSV, JSON, figures, Markdown report, and output zip.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()

candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    Path("/content/mod30-residue-lanes"),
]

REPO_ROOT = None
for c in candidates:
    if (c / "notebooks").exists():
        REPO_ROOT = c
        break

if REPO_ROOT is None:
    REPO_ROOT = cwd

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODULUS = 30
ADMISSIBLE_RESIDUES = [1, 7, 11, 13, 17, 19, 23, 29]
TARGET_LANE = 23
LANE_LABEL = f"{TARGET_LANE:02d}"

print("REPO_ROOT:", REPO_ROOT)
print("TARGET_LANE:", LANE_LABEL)


## Helpers

In [ ]:
def prime_sieve(n_max):
    sieve = np.ones(n_max + 1, dtype=bool)
    sieve[:2] = False

    limit = int(np.sqrt(n_max)) + 1
    for p in range(2, limit):
        if sieve[p]:
            sieve[p*p:n_max+1:p] = False

    return sieve


def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)


def safe_corrcoef(x):
    out = np.corrcoef(x, rowvar=False)
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
    return out


## Generate rolling prime lane vectors

Rows are rolling windows. Columns are the eight admissible residue lanes.


In [ ]:
N_MAX = 150000
WINDOW_SIZE = 3000
STEP_SIZE = 300

values = np.arange(1, N_MAX + 1)
prime_mask = prime_sieve(N_MAX)

df = pd.DataFrame({
    "n": values,
    "residue": values % MODULUS,
})

df["is_admissible"] = df["residue"].isin(ADMISSIBLE_RESIDUES)
df["is_prime"] = prime_mask[values]
df["valid"] = df["is_admissible"] & df["is_prime"]

lane_cols = [f"lane_{r:02d}" for r in ADMISSIBLE_RESIDUES]

rows = []

for start in range(1, N_MAX - WINDOW_SIZE + 2, STEP_SIZE):
    stop = start + WINDOW_SIZE - 1
    window = df[(df["n"] >= start) & (df["n"] <= stop)]
    prime_window = window[window["valid"]]

    counts = (
        prime_window
        .groupby("residue")
        .size()
        .reindex(ADMISSIBLE_RESIDUES, fill_value=0)
    )

    row = {
        "window_start": start,
        "window_stop": stop,
        "window_center": (start + stop) / 2,
        "prime_count": int(prime_window.shape[0]),
    }

    for residue, count in counts.items():
        row[f"lane_{residue:02d}"] = int(count)

    rows.append(row)

rolling_df = pd.DataFrame(rows)

rolling_csv = RESULTS_DIR / "notebook23_rolling_prime_lane_vectors.csv"
rolling_df.to_csv(rolling_csv, index=False)

print(rolling_csv)
rolling_df.head()


## Construct lane similarity graph

Nodes are residue lanes.

Edges are weighted by a blend of:

- Pearson correlation of rolling lane trajectories,
- cosine similarity of rolling lane trajectories,
- normalized covariance strength.


In [ ]:
X = rolling_df[lane_cols].to_numpy(dtype=float)
X_centered = X - X.mean(axis=0, keepdims=True)

corr = safe_corrcoef(X)
cov = np.cov(X_centered, rowvar=False)

cosine = np.zeros((len(lane_cols), len(lane_cols)))
for i in range(len(lane_cols)):
    for j in range(len(lane_cols)):
        cosine[i, j] = cosine_similarity(X[:, i], X[:, j])

cov_abs = np.abs(cov)
cov_norm = cov_abs / max(1e-12, cov_abs.max())

# Convert correlation from [-1, 1] to [0, 1].
corr_scaled = (corr + 1) / 2

adjacency = 0.45 * corr_scaled + 0.45 * cosine + 0.10 * cov_norm
np.fill_diagonal(adjacency, 0)

labels = [f"{r:02d}" for r in ADMISSIBLE_RESIDUES]

corr_df = pd.DataFrame(corr, index=labels, columns=labels)
cosine_df = pd.DataFrame(cosine, index=labels, columns=labels)
cov_df = pd.DataFrame(cov, index=labels, columns=labels)
adj_df = pd.DataFrame(adjacency, index=labels, columns=labels)

corr_csv = RESULTS_DIR / "notebook23_lane_correlation_matrix.csv"
cosine_csv = RESULTS_DIR / "notebook23_lane_cosine_matrix.csv"
cov_csv = RESULTS_DIR / "notebook23_lane_covariance_matrix.csv"
adj_csv = RESULTS_DIR / "notebook23_graph_adjacency_matrix.csv"

corr_df.to_csv(corr_csv)
cosine_df.to_csv(cosine_csv)
cov_df.to_csv(cov_csv)
adj_df.to_csv(adj_csv)

print(adj_csv)
adj_df


## Graph nodes and edges

In [ ]:
degree = adjacency.sum(axis=1)

node_rows = []
for i, residue in enumerate(ADMISSIBLE_RESIDUES):
    strongest_idx = int(np.argmax(adjacency[i]))
    node_rows.append({
        "lane": f"{residue:02d}",
        "residue": residue,
        "weighted_degree": float(degree[i]),
        "mean_prime_count": float(X[:, i].mean()),
        "std_prime_count": float(X[:, i].std(ddof=0)),
        "strongest_neighbor": labels[strongest_idx],
        "strongest_neighbor_weight": float(adjacency[i, strongest_idx]),
        "is_target_lane": residue == TARGET_LANE,
    })

nodes_df = pd.DataFrame(node_rows)

edge_rows = []
for i in range(len(ADMISSIBLE_RESIDUES)):
    for j in range(i + 1, len(ADMISSIBLE_RESIDUES)):
        edge_rows.append({
            "source": labels[i],
            "target": labels[j],
            "weight": float(adjacency[i, j]),
            "correlation": float(corr[i, j]),
            "cosine_similarity": float(cosine[i, j]),
            "covariance": float(cov[i, j]),
        })

edges_df = pd.DataFrame(edge_rows).sort_values("weight", ascending=False)

nodes_csv = RESULTS_DIR / "notebook23_lane_graph_nodes.csv"
edges_csv = RESULTS_DIR / "notebook23_lane_graph_edges.csv"

nodes_df.to_csv(nodes_csv, index=False)
edges_df.to_csv(edges_csv, index=False)

print(nodes_csv)
print(edges_csv)

nodes_df


## Graph Laplacian and embedding

The unnormalized graph Laplacian is:

```text
L = D - A
```

where `D` is the weighted degree matrix and `A` is the adjacency matrix.


In [ ]:
D = np.diag(degree)
L = D - adjacency

eigvals, eigvecs = np.linalg.eigh(L)
order = np.argsort(eigvals)
eigvals = eigvals[order]
eigvecs = eigvecs[:, order]

# Laplacian embedding uses first nontrivial eigenvectors.
embed_x = eigvecs[:, 1]
embed_y = eigvecs[:, 2]

embedding_df = pd.DataFrame({
    "lane": labels,
    "residue": ADMISSIBLE_RESIDUES,
    "laplacian_x": embed_x,
    "laplacian_y": embed_y,
    "weighted_degree": degree,
})

lap_df = pd.DataFrame(L, index=labels, columns=labels)
lap_csv = RESULTS_DIR / "notebook23_graph_laplacian_matrix.csv"
embedding_csv = RESULTS_DIR / "notebook23_laplacian_embedding.csv"

lap_df.to_csv(lap_csv)
embedding_df.to_csv(embedding_csv, index=False)

eigen_rows = []
for k in range(len(eigvals)):
    row = {
        "graph_mode": k,
        "eigenvalue": float(eigvals[k]),
    }
    for i, label in enumerate(labels):
        row[f"loading_{label}"] = float(eigvecs[i, k])
    eigen_rows.append(row)

graph_modes_df = pd.DataFrame(eigen_rows)

graph_modes_csv = RESULTS_DIR / "notebook23_graph_laplacian_modes.csv"
graph_modes_df.to_csv(graph_modes_csv, index=False)

print(lap_csv)
print(embedding_csv)
print(graph_modes_csv)

embedding_df


## Community split from Laplacian mode 2

A simple graph community split can be obtained from the sign of the first nontrivial Laplacian eigenvector.


In [ ]:
community_sign = np.sign(eigvecs[:, 1])
community_sign[community_sign == 0] = 1

nodes_df["laplacian_community"] = ["A" if s >= 0 else "B" for s in community_sign]
nodes_df.to_csv(nodes_csv, index=False)

community_summary = (
    nodes_df
    .groupby("laplacian_community")
    .agg(
        lanes=("lane", lambda x: ", ".join(x)),
        mean_weighted_degree=("weighted_degree", "mean"),
        count=("lane", "size"),
    )
    .reset_index()
)

community_csv = RESULTS_DIR / "notebook23_graph_community_summary.csv"
community_summary.to_csv(community_csv, index=False)

print(community_csv)
community_summary


## Graph signal modes

Project rolling lane vectors onto graph Laplacian eigenmodes.


In [ ]:
graph_scores = X_centered @ eigvecs

score_rows = []
for i, row in rolling_df.iterrows():
    out = {
        "window_start": int(row["window_start"]),
        "window_center": float(row["window_center"]),
    }
    for k in range(min(4, graph_scores.shape[1])):
        out[f"graph_mode_{k}_score"] = float(graph_scores[i, k])
    score_rows.append(out)

graph_scores_df = pd.DataFrame(score_rows)

graph_scores_csv = RESULTS_DIR / "notebook23_graph_signal_mode_scores.csv"
graph_scores_df.to_csv(graph_scores_csv, index=False)

print(graph_scores_csv)
graph_scores_df.head()


## Summary exports

In [ ]:
summary = {
    "repo": "mod30-residue-lanes",
    "notebook": "23_lane_residue_23",
    "title": "Graph Embeddings of Residue-Lane Manifolds",
    "modulus": MODULUS,
    "target_lane": TARGET_LANE,
    "target_lane_label": LANE_LABEL,
    "admissible_residues": ADMISSIBLE_RESIDUES,
    "n_max": N_MAX,
    "window_size": WINDOW_SIZE,
    "step_size": STEP_SIZE,
    "rolling_windows": int(len(rolling_df)),
    "graph_nodes": int(len(nodes_df)),
    "graph_edges": int(len(edges_df)),
    "mean_edge_weight": float(edges_df["weight"].mean()),
    "max_edge_weight": float(edges_df["weight"].max()),
    "min_edge_weight": float(edges_df["weight"].min()),
    "strongest_edge": {
        "source": str(edges_df.iloc[0]["source"]),
        "target": str(edges_df.iloc[0]["target"]),
        "weight": float(edges_df.iloc[0]["weight"]),
    },
    "target_lane_weighted_degree": float(nodes_df.loc[nodes_df["lane"] == LANE_LABEL, "weighted_degree"].iloc[0]),
    "target_lane_strongest_neighbor": str(nodes_df.loc[nodes_df["lane"] == LANE_LABEL, "strongest_neighbor"].iloc[0]),
    "laplacian_eigenvalues": [float(x) for x in eigvals],
    "constraint_view": "Residue lanes become a graph manifold when rolling prime-lane trajectories define weighted relationships between lanes.",
}

json_path = RESULTS_DIR / "notebook23_graph_embedding_summary.json"
json_path.write_text(json.dumps(summary, indent=2))

print(json_path)
print(json.dumps(summary, indent=2))


## Figures

In [ ]:
figure_names = []

def savefig(name):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    figure_names.append(name)
    print(path)

# 1. Adjacency matrix
plt.figure(figsize=(7, 6))
plt.imshow(adjacency)
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)
plt.colorbar(label="Edge weight")
plt.title("Notebook 23 — Graph Adjacency Matrix")
savefig("notebook23_graph_adjacency_matrix.png")

# 2. Correlation matrix
plt.figure(figsize=(7, 6))
plt.imshow(corr, vmin=-1, vmax=1)
plt.xticks(range(len(labels)), labels)
plt.yticks(range(len(labels)), labels)
plt.colorbar(label="Correlation")
plt.title("Notebook 23 — Lane Correlation Matrix")
savefig("notebook23_lane_correlation_matrix.png")

# 3. Laplacian embedding
plt.figure(figsize=(8, 6))
plt.scatter(embedding_df["laplacian_x"], embedding_df["laplacian_y"], s=220)
for _, r in embedding_df.iterrows():
    plt.text(r["laplacian_x"], r["laplacian_y"], r["lane"], ha="center", va="center", fontsize=11)
plt.xlabel("Laplacian embedding x")
plt.ylabel("Laplacian embedding y")
plt.title("Notebook 23 — Laplacian Embedding of Residue Lanes")
savefig("notebook23_laplacian_embedding.png")

# 4. Weighted degree
plt.figure(figsize=(8, 4))
plt.bar(nodes_df["lane"], nodes_df["weighted_degree"])
plt.xlabel("Residue lane")
plt.ylabel("Weighted degree")
plt.title("Notebook 23 — Weighted Degree Centrality")
savefig("notebook23_node_centrality.png")

# 5. Edge weights sorted
plt.figure(figsize=(10, 4))
edge_labels = edges_df["source"] + "–" + edges_df["target"]
plt.bar(edge_labels, edges_df["weight"])
plt.xticks(rotation=70, ha="right")
plt.xlabel("Edge")
plt.ylabel("Weight")
plt.title("Notebook 23 — Sorted Graph Edge Weights")
savefig("notebook23_sorted_edge_weights.png")

# 6. Graph signal modes
for mode in range(1, 4):
    plt.figure(figsize=(8, 4))
    plt.bar(labels, eigvecs[:, mode])
    plt.axhline(0, linewidth=1)
    plt.xlabel("Residue lane")
    plt.ylabel("Laplacian loading")
    plt.title(f"Notebook 23 — Graph Mode {mode} Loadings")
    savefig(f"notebook23_graph_mode_{mode}_loadings.png")

# 7. Graph mode scores
plt.figure(figsize=(12, 5))
for k in range(1, 4):
    plt.plot(graph_scores_df["window_center"], graph_scores_df[f"graph_mode_{k}_score"], label=f"graph mode {k}")
plt.xlabel("Window center")
plt.ylabel("Graph mode score")
plt.title("Notebook 23 — Graph Signal Mode Scores")
plt.legend()
savefig("notebook23_graph_signal_mode_scores.png")

# 8. Similarity graph as weighted chords in embedding
plt.figure(figsize=(8, 6))
coords = embedding_df.set_index("lane")[["laplacian_x", "laplacian_y"]].to_dict(orient="index")
w_min = edges_df["weight"].min()
w_max = edges_df["weight"].max()

for _, e in edges_df.iterrows():
    x0, y0 = coords[e["source"]]["laplacian_x"], coords[e["source"]]["laplacian_y"]
    x1, y1 = coords[e["target"]]["laplacian_x"], coords[e["target"]]["laplacian_y"]
    width = 0.5 + 3.0 * (e["weight"] - w_min) / max(1e-12, w_max - w_min)
    plt.plot([x0, x1], [y0, y1], alpha=0.35, linewidth=width)

plt.scatter(embedding_df["laplacian_x"], embedding_df["laplacian_y"], s=260)
for _, r in embedding_df.iterrows():
    plt.text(r["laplacian_x"], r["laplacian_y"], r["lane"], ha="center", va="center", fontsize=11)

plt.xlabel("Laplacian embedding x")
plt.ylabel("Laplacian embedding y")
plt.title("Notebook 23 — Lane Similarity Graph")
savefig("notebook23_lane_similarity_graph.png")


## Build Markdown report

The report uses repo-relative links for CSV, JSON, and figure outputs.


In [ ]:
report_path = REPORTS_DIR / "report_23_graph_embeddings.md"

output_links = "\n".join([
    '- Rolling prime lane vectors CSV: <a href="../results/notebook23_rolling_prime_lane_vectors.csv">`results/notebook23_rolling_prime_lane_vectors.csv`</a>',
    '- Graph nodes CSV: <a href="../results/notebook23_lane_graph_nodes.csv">`results/notebook23_lane_graph_nodes.csv`</a>',
    '- Graph edges CSV: <a href="../results/notebook23_lane_graph_edges.csv">`results/notebook23_lane_graph_edges.csv`</a>',
    '- Adjacency matrix CSV: <a href="../results/notebook23_graph_adjacency_matrix.csv">`results/notebook23_graph_adjacency_matrix.csv`</a>',
    '- Laplacian matrix CSV: <a href="../results/notebook23_graph_laplacian_matrix.csv">`results/notebook23_graph_laplacian_matrix.csv`</a>',
    '- Laplacian embedding CSV: <a href="../results/notebook23_laplacian_embedding.csv">`results/notebook23_laplacian_embedding.csv`</a>',
    '- Graph modes CSV: <a href="../results/notebook23_graph_laplacian_modes.csv">`results/notebook23_graph_laplacian_modes.csv`</a>',
    '- Graph signal mode scores CSV: <a href="../results/notebook23_graph_signal_mode_scores.csv">`results/notebook23_graph_signal_mode_scores.csv`</a>',
    '- Summary JSON: <a href="../results/notebook23_graph_embedding_summary.json">`results/notebook23_graph_embedding_summary.json`</a>',
] + [
    f'- Figure: <a href="../figures/{name}">`figures/{name}`</a>'
    for name in figure_names
])

report = f"""# Report 23 — Graph Embeddings of Residue-Lane Manifolds

Notebook 23 treats residue lanes as graph nodes and rolling prime-lane correlations as weighted edges.

Constraint view:

> Residue lanes become a graph manifold when rolling prime-lane trajectories define weighted relationships between lanes.

## Generated outputs

{output_links}

## Summary

| Metric | Value |
|---|---:|
| Modulus | {MODULUS} |
| Target lane | {LANE_LABEL} |
| Rolling windows | {len(rolling_df)} |
| Graph nodes | {len(nodes_df)} |
| Graph edges | {len(edges_df)} |
| Mean edge weight | {edges_df["weight"].mean():.6f} |
| Max edge weight | {edges_df["weight"].max():.6f} |
| Min edge weight | {edges_df["weight"].min():.6f} |
| Target lane weighted degree | {nodes_df.loc[nodes_df["lane"] == LANE_LABEL, "weighted_degree"].iloc[0]:.6f} |

## Graph nodes

{nodes_df.to_markdown(index=False)}

## Strongest graph edges

{edges_df.head(12).to_markdown(index=False)}

## Community summary

{community_summary.to_markdown(index=False)}

## Interpretation

- Notebook 19 studied temporal spectral dynamics.
- Notebook 23 converts rolling lane trajectories into a weighted graph.
- Nodes are admissible residue lanes.
- Edges encode trajectory similarity, cosine alignment, and covariance strength.
- Laplacian embedding turns lane relationships into a learned graph geometry.
- Graph signal modes describe how rolling prime-count activity moves across graph eigenmodes.

## Next step

Notebook 29 can study sparse feature emergence and boundary-reset behavior on top of this graph manifold.
"""

report_path.write_text(report)
print(report_path)


## Report preview

In [ ]:
print(report_path.read_text()[:5000])


## Render generated figures in notebook

In [ ]:
from IPython.display import Image, display, Markdown

for name in figure_names:
    display(Markdown(f"### `{name}`"))
    display(Image(filename=str(FIGURES_DIR / name)))


## Create output zip

In [ ]:
EXPORT_NAME = "notebook23_graph_embeddings_outputs.zip"
export_path = REPO_ROOT / EXPORT_NAME

with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in RESULTS_DIR.glob("notebook23_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in FIGURES_DIR.glob("notebook23_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
    for p in REPORTS_DIR.glob("report_23_*"):
        zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))

print(export_path)


## Optional Colab download

Uncomment and run this cell in Colab to download generated outputs.


In [ ]:
# OPTIONAL COLAB DOWNLOAD
#
# EXPORT_NAME = "notebook23_graph_embeddings_outputs.zip"
# export_path = REPO_ROOT / EXPORT_NAME
#
# with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as zf:
#     for folder in [RESULTS_DIR, FIGURES_DIR, REPORTS_DIR]:
#         for p in folder.glob("notebook23_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#         for p in folder.glob("report_23_*"):
#             zf.write(p, arcname=str(p.relative_to(REPO_ROOT)))
#
# from google.colab import files
# files.download(str(export_path))
